# Group 3

In [52]:
!pip install ftfy regex tqdm torchmetrics git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-ascq8ork
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-ascq8ork
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done


In [53]:
import os

# Define paths
zip_path = './aml-group-project.zip'
extract_to = './AMLdataset'

# Automated Unzip
if not os.path.exists(extract_to):
    os.makedirs(extract_to)
    print(f"Unzipping {zip_path}...")

    # Using the shell command is usually faster for large zips
    !unzip -q {zip_path} -d {extract_to}

    print("Unzip complete!")
else:
    print("Dataset already unzipped.")

Dataset already unzipped.


In [54]:
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from PIL import Image
from torch.utils.data import Dataset


class AMLDataset(Dataset):
    def __init__(self, csv_path, imgs_dir, train=True, transform=None):
        self.imgs_dir = imgs_dir
        self.train = train
        self.transform = transform

        full_df = pd.read_csv(csv_path)

        train_df, test_df = train_test_split(
            full_df,
            test_size=0.20,            # 20% for validation
            random_state=42,
            stratify=full_df['label']  # Ensures all 250 classes are in both
        )

        self.df = (train_df if train else test_df).reset_index(drop=True)

        # Create the 'classes' attribute (Unique list of names)
        # We sort them to ensure the index mapping is always consistent
        self.classes = sorted(self.df['label'].unique().tolist())

        # Create a mapping from Name -> Integer ID
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # Create the 'labels' attribute (The ID for every single row)
        # This is helpful if you want to use a Weighted or Balanced Sampler later
        self.labels = [self.class_to_idx[name] for name in self.df['label']]


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_img(row['filename'])
        return img, self.labels[idx]

    def _load_img(self, filename):
        path = os.path.join(self.imgs_dir, str(filename))
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

In [55]:
import os
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset


class AMLValidation(Dataset):
    def __init__(self, csv_path, imgs_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.imgs_dir = imgs_dir
        self.transform = transform

        # Group the rows by episode_id so __len__ returns total number of tasks
        self.episode_ids = sorted(self.df['episode_id'].unique())

    def __len__(self):
        return len(self.episode_ids)

    def load_img(self, filename):
        img_path = os.path.join(self.imgs_dir, str(filename))
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        # Return a full 5-way 5-shot Episode
        ep_id = self.episode_ids[idx]
        ep_df = self.df[self.df['episode_id'] == ep_id]

        support_df = ep_df[ep_df['role'] == 'support']
        query_df = ep_df[ep_df['role'] == 'query']

        # Pack support images and their labels
        s_imgs = torch.stack([self.load_img(f) for f in support_df['filename']])
        s_labels = torch.tensor(support_df['label'].values, dtype=torch.long)

        # Pack query images
        q_imgs = torch.stack([self.load_img(f) for f in query_df['filename']])

        # If your test CSV has query labels (sometimes they are hidden), include them:
        # q_labels = torch.tensor(query_df['label'].values) if 'label' in query_df.columns else None

        return {
            "support_imgs": s_imgs,
            "support_labels": s_labels,
            "query_imgs": q_imgs,
            "episode_id": ep_id
        }

In [56]:
import os
import clip
import torch
import torchmetrics
import pandas as pd

from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Define constants
LR = 1e-4 # Learning Rate
BATCH_SIZE = 32 # Number of images processed in one iteration
SHOTS = 5 # 5-way 5-shot
EPOCHS = 5 # Number of epochs

CKPT_PATH  = "best_student.pt"
CSV_PATH  = "./AMLdataset/release/train.csv"
IMGS_DIR  = "./AMLdataset/release/images/"
VAL_CSV   = "./AMLdataset/release/test_episodes_release.csv"

# Define controls
best_acc = 0.0 # Used to save best checkpoints

# Load the models
device = "cuda" if torch.cuda.is_available() else "cpu"
student, preprocess = clip.load('ViT-B/32', device, jit=False)
teacher, _ = clip.load('ViT-L/14', device, jit=False)
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False

# Download the dataset
train_dataset = AMLDataset(
    csv_path=CSV_PATH,
    imgs_dir=IMGS_DIR,
    train=True,
    transform=preprocess
)

test_dataset = AMLDataset(
    csv_path=CSV_PATH,
    imgs_dir=IMGS_DIR,
    train=False,
    transform=preprocess
)

val_dataset = AMLValidation(
    csv_path=VAL_CSV,
    imgs_dir=IMGS_DIR,
    transform=preprocess
)

# Prepare the data
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True,     # Helpful for CLIP to keep matrix dimensions consistent
    pin_memory=True     # Helpful for GPU
)
num_batches_train = len(train_dataloader)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True,     # Helpful for CLIP to keep matrix dimensions consistent
    pin_memory=True     # Helpful for GPU
)
num_batches_test = len(test_dataloader)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_val = len(val_dataloader)

all_class_texts = torch.cat([clip.tokenize(f"a photo of a {c}") for c in train_dataset.classes]).to(device)
NUM_CLASSES = len(train_dataset.classes)

# Prepare criterion and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in student.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.1
)

# Cosine schedule: smoothly decays LR to 0 over all training steps
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS * len(train_dataloader)
)

# Distillation temperature
# Higher T -> softer teacher distribution -> gentler supervision signal
KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.1   # weight of distillation loss vs contrastive loss

In [61]:
def train_one_epoch(epoch: int) -> float:
    student.train()
    epoch_loss = 0.0

    for images, class_ids in tqdm(train_dataloader, total=num_batches_train, desc=f"[Train] Epoch {epoch}"):
        images = images.to(device)

        # Build paired text tokens for this batch
        # Each image gets the text of its own class - this is what CLIP aligns
        texts = clip.tokenize([
            f"a photo of a {train_dataset.classes[i]}"
            for i in class_ids
        ]).to(device)

        optimizer.zero_grad()

        # Student forward
        # logits_per_image[i, j] = similarity(image_i, text_j) * scale
        # Ground truth: diagonal (image i matches text i)
        logits_per_image, logits_per_text = student(images, texts)

        ground_truth = torch.arange(len(images), dtype=torch.long, device=device)

        # Symmetric contrastive loss (standard CLIP objective)
        contrastive_loss = (
            criterion(logits_per_image, ground_truth) +
            criterion(logits_per_text,  ground_truth)
        ) / 2

        # Knowledge Distillation loss
        # Teacher produces a soft similarity distribution over the batch.
        # Student is trained to match it - transfers teacher's "uncertainty"
        # about near-duplicate classes rather than forcing hard 0/1 targets.
        with torch.no_grad():
            t_logits_img, t_logits_txt = teacher(images, texts)

        # Soft targets via temperature scaling
        soft_targets_img = torch.softmax(t_logits_img / KD_TEMPERATURE, dim=-1)
        soft_targets_txt = torch.softmax(t_logits_txt / KD_TEMPERATURE, dim=-1)

        # KL divergence: how far is student distribution from teacher's?
        kd_loss = (
            torch.nn.functional.kl_div(
                torch.log_softmax(logits_per_image / KD_TEMPERATURE, dim=-1),
                soft_targets_img, reduction='batchmean'
            ) +
            torch.nn.functional.kl_div(
                torch.log_softmax(logits_per_text / KD_TEMPERATURE, dim=-1),
                soft_targets_txt, reduction='batchmean'
            )
        ) / 2 * (KD_TEMPERATURE ** 2)  # T² rescaling keeps gradient magnitude stable

        total_loss = (1 - KD_ALPHA) * contrastive_loss + KD_ALPHA * kd_loss

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += total_loss

    return epoch_loss / len(train_dataloader)

In [63]:
def evaluate_test() -> float:
    """
    Linear probe evaluation: encode all test images, compute cosine similarity
    against all class text prototypes, pick argmax.
    This is the standard zero-shot CLIP evaluation style, applied to our classes.
    """
    student.eval()

    # Pre-compute text prototypes for all classes once
    with torch.no_grad():
        text_tokens = clip.tokenize([
            f"a photo of a {c}" for c in train_dataset.classes
        ]).to(device)
        # shape: [NUM_CLASSES, D]
        text_feats = student.encode_text(text_tokens)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

    correct = 0
    total   = 0

    with torch.no_grad():
        for images, labels in tqdm(test_dataloader, total=num_batches_test, desc="[Test]"):
            images = images.to(device)
            labels = labels.to(device)

            # shape: [B, D]
            img_feats = student.encode_image(images)
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)

            # Cosine similarity matrix: [B, NUM_CLASSES]
            # The 100x scale matches CLIP's internal logit scale
            sims = (img_feats @ text_feats.T) * 100.0

            preds = sims.argmax(dim=-1)
            correct += (preds == labels).sum()
            total   += labels.size(0)

    return correct / total

In [59]:
def evaluate_val() -> dict:
    """
    Prototypical network inference over pre-defined episodes.
    No query labels available — returns predictions keyed by episode_id
    for submission instead of computing accuracy.
    """
    student.eval()
    predictions = {}  # {episode_id: [pred_0, pred_1, ...]}

    with torch.no_grad():
        for batch in tqdm(val_dataloader, total=num_batches_val, desc="[Val]"):
            s_imgs   = batch['support_imgs'].squeeze(0).to(device)
            s_labels = batch['support_labels'].squeeze(0).to(device)
            q_imgs   = batch['query_imgs'].squeeze(0).to(device)
            episode_id = batch['episode_id'].item()

            # Encode
            s_feats = student.encode_image(s_imgs)
            s_feats = s_feats / s_feats.norm(dim=-1, keepdim=True)

            q_feats = student.encode_image(q_imgs)
            q_feats = q_feats / q_feats.norm(dim=-1, keepdim=True)

            # Build prototypes
            n_classes  = s_labels.max().item() + 1
            embed_dim  = s_feats.shape[-1]
            prototypes = torch.zeros(n_classes, embed_dim, device=device)

            for cls_id in range(n_classes):
                mask = (s_labels == cls_id)
                prototypes[cls_id] = s_feats[mask].mean(dim=0)

            prototypes = prototypes / prototypes.norm(dim=-1, keepdim=True)

            # Classify queries — local class indices (0..N-1)
            sims  = q_feats @ prototypes.T
            preds = sims.argmax(dim=-1).cpu().tolist()

            predictions[episode_id] = preds

    return predictions

In [ ]:
best_acc = 0.0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(epoch)

    test_acc   = evaluate_test()   # standard classification accuracy on held-out split

    print(f"Epoch {epoch:02d} | Loss: {train_loss:.4f} | Test Acc: {test_acc:.4f}")

    # Checkpoint on test accuracy (best proxy you have without val labels)
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': student.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'test_acc': test_acc,
        }, CKPT_PATH)
        print(f"Checkpoint saved (test_acc={test_acc:.4f})")

# After training, run final inference and save predictions
val_preds = evaluate_val()
pd.DataFrame([
    {"episode_id": ep_id, "query_idx": i, "predicted_label": label}
    for ep_id, preds in val_preds.items()
    for i, label in enumerate(preds)
]).to_csv("val_predictions.csv", index=False)
print("Predictions saved to val_predictions.csv")

[Test]: 100%|██████████| 31/31 [00:04<00:00,  6.72it/s]


Epoch 00 | Loss: nan | Test Acc: 0.0040
Checkpoint saved (test_acc=0.0040)


[Train] Epoch 1:  22%|██▏       | 28/125 [00:08<00:26,  3.69it/s]

In [ ]:
def evaluate_episodic(model, dataloader, device):
    model.eval()
    all_episode_accs = []

    print("Running Episodic Evaluation...")
    with torch.no_grad():
        for episode in tqdm(dataloader):
            # Setup (batch_size is 1, so we squeeze)
            s_imgs = episode["support_imgs"].squeeze(0).to(device)   # [25, 3, 224, 224]
            s_labels = episode["support_labels"].squeeze(0).to(device)
            q_imgs = episode["query_imgs"].squeeze(0).to(device)     # [25, 3, 224, 224]

            # Get Features
            s_feats = model.encode_image(s_imgs)
            q_feats = model.encode_image(q_imgs)

            # Normalize
            s_feats /= s_feats.norm(dim=-1, keepdim=True)
            q_feats /= q_feats.norm(dim=-1, keepdim=True)

            # Create Class Prototypes (The "Average" for each of the 5 classes)
            unique_labels = torch.unique(s_labels) # These are your 5 classes for this episode
            prototypes = []
            for label in unique_labels:
                # Find all support images that belong to this specific label and average them
                class_prototype = s_feats[s_labels == label].mean(dim=0)
                prototypes.append(class_prototype / class_prototype.norm())

            prototypes = torch.stack(prototypes) # Shape: [5, 512]

            # Classify Queries against Prototypes
            # [25 queries, 512] @ [512, 5 prototypes] -> [25, 5] similarity matrix
            logits = 100.0 * q_feats @ prototypes.T
            preds = logits.argmax(dim=-1)

            # Accuracy Calculation
            # Note: Few-shot query labels are usually 0-4 (indices of the unique_labels)
            # If your dataset provides true labels for queries, use them here:
            if "query_labels" in episode:
                q_labels = episode["query_labels"].squeeze(0).to(device)
                # Map the true labels to 0-4 range to match our prototypes index
                label_to_idx = {val.item(): i for i, val in enumerate(unique_labels)}
                target_indices = torch.tensor([label_to_idx[l.item()] for l in q_labels], device=device)

                acc = (preds == target_indices).float().mean()
                all_episode_accs.append(acc)

        final_acc = torch.stack(all_episode_accs).mean()
        print(f"\nFinal Episodic Accuracy: {final_acc:.2%}")
        return final_acc


evaluate_episodic(student, val_dataloader, device)